[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_74_AB_Testing_Guarded_Rollouts.ipynb)

# Lesson 74 — A/B Testing & Guarded Rollouts

**Phase 8: Production Ops for LLM Systems · Lesson 4 of ~6**

You have an observability stack now. Lesson 71 gave you **online evals & drift detection**, Lesson 72 gave you **structured logging & trace search**, and Lesson 73 gave you **SLOs, error budgets & burn-rate alerting**. Those tools tell you when something is *already* wrong in production.

This lesson is about the moment you **deliberately introduce risk**: shipping a new prompt, a new model, or a new agent version. The naive move is to flip it on for 100% of traffic and hope. The professional move is a **guarded rollout** — expose the change to a *sliver* of traffic, measure it against the current version with a real statistical test, and let last lesson's alerting **auto-roll-back** the change if it burns your error budget too fast.

> **The one-line mental model:** an A/B test *decides whether the new version is better*; a guarded rollout *makes sure a bad version can never hurt more than a few percent of users before it's pulled.* Today you build both, and you wire yesterday's burn-rate alert in as the rollback trigger.

By the end you'll have a reusable `observability/rollout.py` with sticky bucketing, a two-proportion significance test, and a staged-ramp controller that promotes a good challenger to 100% and rolls back a bad one at 5%.

## Where this sits in Phase 8

| Lesson | Topic | The question it answers |
|---|---|---|
| L71 | Observability & online evals | *Is production quality drifting?* |
| L72 | Structured logging & trace search | *Show me exactly what broke, for whom.* |
| L73 | Alerting, SLOs & on-call | *When should a human get paged?* |
| **L74 (today)** | **A/B testing & guarded rollouts** | ***Is the new version actually better — and how do I ship it without risking everyone?*** |
| L75 (next) | Feedback loops & the data flywheel | *How does production turn into better training/eval data?* |
| L76 | Phase 8 capstone | *Wire it all into `agent-bench` as a prod-obs module.* |

**Champion vs. challenger.** Throughout, the version currently serving production is the **champion** (arm **A** / control). The new thing you want to prove out is the **challenger** (arm **B** / treatment). A rollout is the disciplined process of letting the challenger *earn* the throne.

In [ ]:
# === Setup (no API key needed — this whole lesson is deterministic) ===
# In Colab this installs rich for pretty tables. Everything else is stdlib.
try:
    import rich  # noqa
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "rich", "-q"], check=False)

import os, math, random, hashlib, json
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Callable, Optional
from rich.console import Console
from rich.table import Table

# force_jupyter=False keeps output clean & copy-pasteable in Colab and when
# this notebook is executed head-less during validation (L64 pitfall).
console = Console(force_jupyter=False, no_color=False, highlight=False)

SEED = 74
random.seed(SEED)

# A frozen "now" so any time-based reasoning is 100% reproducible (L72/L73 habit).
NOW = datetime(2026, 7, 20, 12, 0, 0, tzinfo=timezone.utc)

# Where we'll write the reusable module. In Colab this is /content.
BASE = "/content"
os.makedirs(os.path.join(BASE, "observability"), exist_ok=True)

print("Setup OK — seed", SEED, "| now", NOW.isoformat(), "| BASE", BASE)

## 1. Why "just deploy it" is a trap

You changed the system prompt and your offline `agent-bench` (Phase 7) score went up 2 points. Ship it to everyone, right?

Two things can bite you:

1. **Offline ≠ online.** Your eval set is a *sample*. The 2-point gain might be noise, or it might not transfer to the messy real distribution (L71's whole point).
2. **A regression at 100% hits 100% of users instantly.** If the new prompt quietly makes the agent 8% more likely to fail on a request type you under-tested, everyone feels it before your dashboards even finish their first rollup.

| | Naive full deploy | Guarded rollout |
|---|---|---|
| Blast radius of a bad change | **100% of traffic** | 5% (the first stage) |
| "Is it better?" decided by | vibes / a single offline number | a **statistical test** on live traffic |
| Rollback | manual, after users complain | **automatic**, tripped by a burn-rate alert (L73) |
| Assignment of users | — | **sticky** (a user stays in one arm, no flip-flopping) |
| Ship speed | instant but risky | staged: 5% → 25% → 50% → 100% |

The rest of the lesson builds each row of the right-hand column.

## 2. Sticky bucketing — deterministic, hash-based assignment

The first primitive: given a user (or session, or request) and a rollout percentage, decide **A or B** — and make it *sticky* so the same user always lands in the same arm. If a user bounced between the old and new prompt on every request, (a) their experience would be incoherent and (b) your experiment would be contaminated (the same unit in both arms).

The trick is **not** `random.random() < split`. That's not sticky and not reproducible. Instead we **hash the unit id** into a stable number in `[0, 1)` and compare to the split. Same id → same hash → same side of the line, forever. Change the `salt` and you get a *fresh, independent* bucketing (useful for running a second, unrelated experiment).

In [ ]:
def bucket_of(unit_id: str, salt: str = "exp-prompt-v2") -> float:
    # Hash "salt:unit_id" to a stable float in [0, 1). md5 is fine here — we
    # need a uniform spread, not cryptographic strength.
    h = hashlib.md5(f"{salt}:{unit_id}".encode()).hexdigest()
    # take 8 hex chars (32 bits) -> integer -> normalize to [0,1)
    return int(h[:8], 16) / 0xFFFFFFFF

def assign_arm(unit_id: str, split: float, salt: str = "exp-prompt-v2") -> str:
    # split = fraction of traffic sent to the challenger (arm B).
    # Because bucket_of is stable, growing `split` only ever MOVES USERS INTO B,
    # never shuffles existing B users back to A — a ramp is monotonic.
    return "B" if bucket_of(unit_id, salt) < split else "A"

# --- self-test: distribution matches the split, and assignment is sticky ---
N = 20_000
users = [f"user_{i}" for i in range(N)]
frac_B = sum(1 for u in users if assign_arm(u, 0.25) == "B") / N
print(f"Requested split 0.25  ->  measured B share {frac_B:.4f}")
assert abs(frac_B - 0.25) < 0.01, "bucketing should honor the split"

# stickiness: calling twice gives the same answer
assert all(assign_arm(u, 0.25) == assign_arm(u, 0.25) for u in users[:1000])

# monotonic ramp: everyone in B at 10% is still in B at 30%
b_at_10 = {u for u in users if assign_arm(u, 0.10) == "B"}
b_at_30 = {u for u in users if assign_arm(u, 0.30) == "B"}
assert b_at_10 <= b_at_30, "ramping up must never eject an existing B user"
print(f"Ramp monotonic: {len(b_at_10)} users in B@10% all still in B@30% "
      f"({len(b_at_30)} total). ✔")

# 💡 EXPERIMENT: change salt to "exp-model-swap" and watch which users land in B
# completely reshuffle — that's how you run two independent experiments at once.

## 3. A deterministic two-arm traffic simulator

To test a rollout controller we need traffic where we *know the ground truth*. We'll simulate requests where:

- **Arm A (champion)** succeeds ~96% of the time with a modest latency distribution.
- **Arm B (challenger)** is configured per-scenario. We'll build **two challengers**:
  - a **bad** one (success drops to ~88%, latency inflates) — the controller *must roll this back*, and
  - a **good** one (success rises to ~98%, latency improves) — the controller *should promote it to 100%*.

Everything is seeded, so the "true" arm qualities are baked in and the controller's job is to *detect* them from noisy samples — exactly the real problem.

In [ ]:
@dataclass
class Event:
    unit_id: str
    arm: str
    success: bool
    latency_ms: float
    cost_usd: float

@dataclass
class ArmSpec:
    name: str
    p_success: float       # true success probability
    lat_mu: float          # lognormal mu for latency
    lat_sigma: float
    cost: float

CHAMPION   = ArmSpec("A-champion", 0.96, math.log(800),  0.35, 0.0030)
BAD_CHAL   = ArmSpec("B-bad",      0.88, math.log(1100), 0.55, 0.0032)  # regression
GOOD_CHAL  = ArmSpec("B-good",     0.98, math.log(700),  0.30, 0.0031)  # improvement

def _draw(spec: ArmSpec, rng: random.Random, unit_id: str) -> Event:
    success = rng.random() < spec.p_success
    latency = math.exp(rng.gauss(spec.lat_mu, spec.lat_sigma))
    return Event(unit_id, spec.name, success, latency, spec.cost)

def simulate_stage(champion: ArmSpec, challenger: ArmSpec, split: float,
                   n_requests: int, stage_idx: int) -> list:
    # Fresh, stage-seeded RNG so each stage is reproducible & independent.
    rng = random.Random(SEED * 1000 + stage_idx)
    events = []
    for i in range(n_requests):
        # a rotating pool of users so sticky assignment actually matters
        unit = f"user_{(stage_idx * n_requests + i) % 5000}"
        arm = assign_arm(unit, split)
        spec = challenger if arm == "B" else champion
        events.append(_draw(spec, rng, unit))
    return events

# quick smoke: at 50% split the two arms get roughly equal traffic
ev = simulate_stage(CHAMPION, BAD_CHAL, 0.50, 4000, stage_idx=0)
nA = sum(1 for e in ev if e.arm.startswith("A"))
nB = sum(1 for e in ev if e.arm.startswith("B"))
print(f"50% split over {len(ev)} reqs -> A={nA}  B={nB}")
assert 0.45 < nB / len(ev) < 0.55
print("Traffic simulator OK ✔")

## 4. Is B actually better? A two-proportion significance test

Suppose at some stage arm A had 40 failures in 1000 requests (96.0% success) and arm B had 34 failures in 350 requests (90.3% success). Is B *really* worse, or did we just get unlucky in a small sample?

This is the classic **two-proportion z-test**. We ask: *if the two arms had identical true success rates, how surprising is a gap this big?* That surprise is the **p-value**. Small p (< 0.05 by convention) → the gap is unlikely to be noise → the difference is **statistically significant**.

We also report a **confidence interval** on the difference `p_B − p_A`. The CI is often more useful than the p-value: it tells you not just "different" but *how much* and *how precisely*.

We implement the normal CDF from `math.erf` so there are no scipy dependencies.

In [ ]:
def normal_cdf(x: float) -> float:
    # Phi(x): probability a standard normal is <= x.
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

@dataclass
class TestResult:
    p_a: float
    p_b: float
    diff: float          # p_b - p_a
    z: float
    p_value: float       # two-sided
    ci_low: float
    ci_high: float
    n_a: int
    n_b: int

def two_proportion_ztest(succ_a: int, n_a: int, succ_b: int, n_b: int,
                         z_crit: float = 1.96) -> TestResult:
    p_a = succ_a / n_a
    p_b = succ_b / n_b
    # pooled proportion under H0 (arms identical) for the z statistic
    p_pool = (succ_a + succ_b) / (n_a + n_b)
    se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se_pool if se_pool > 0 else 0.0
    p_value = 2 * (1 - normal_cdf(abs(z)))          # two-sided
    # unpooled SE for the CI on the difference
    se_diff = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    diff = p_b - p_a
    return TestResult(p_a, p_b, diff, z, p_value,
                      diff - z_crit * se_diff, diff + z_crit * se_diff, n_a, n_b)

# worked example from the markdown
r = two_proportion_ztest(succ_a=960, n_a=1000, succ_b=316, n_b=350)
print(f"A success {r.p_a:.3f} (n={r.n_a}) | B success {r.p_b:.3f} (n={r.n_b})")
print(f"diff (B-A) = {r.diff:+.3f}   95% CI [{r.ci_low:+.3f}, {r.ci_high:+.3f}]")
print(f"z = {r.z:.2f}   p-value = {r.p_value:.4f}")
assert r.p_value < 0.05, "a ~6pt drop on these sample sizes should be significant"
assert r.ci_high < 0, "the whole CI is below zero -> B is genuinely worse"
print("\nStat test OK — B is significantly worse here ✔")

## 5. The peeking problem (why you can't just refresh the dashboard)

Here's the mistake that ruins real experiments: you run the test, check the p-value every few minutes, and **stop the moment it dips below 0.05**. That feels responsible. It is statistically broken.

Each check is another roll of the dice at a false positive. If you peek 20 times at an experiment where the two arms are *truly identical* (an "A/A test"), your chance of seeing a spurious "significant!" result at least once is far above 5% — often 20–40%. You'll "discover" effects that aren't there.

Let's prove it, then name the fixes.

In [ ]:
def run_aa_experiment_with_peeking(rng: random.Random, n_checks: int,
                                    step: int, p_true: float = 0.95) -> bool:
    # A/A test: BOTH arms have the identical true success rate p_true.
    # We add `step` samples per arm between each of n_checks peeks and declare
    # "significant" if p<0.05 is EVER seen. Returns True if we falsely did.
    sa = na = sb = nb = 0
    for _ in range(n_checks):
        for _ in range(step):
            sa += rng.random() < p_true; na += 1
            sb += rng.random() < p_true; nb += 1
        if na > 30 and nb > 30:
            if two_proportion_ztest(sa, na, sb, nb).p_value < 0.05:
                return True     # stopped early on a false positive
    return False

TRIALS = 400
peek_rng = random.Random(999)
false_pos_peek = sum(run_aa_experiment_with_peeking(peek_rng, n_checks=20, step=50)
                     for _ in range(TRIALS)) / TRIALS

# contrast: decide the sample size ONCE, look ONCE at the end
fixed_rng = random.Random(999)
false_pos_fixed = sum(run_aa_experiment_with_peeking(fixed_rng, n_checks=1, step=1000)
                      for _ in range(TRIALS)) / TRIALS

print(f"A/A false-positive rate WITH 20x peeking : {false_pos_peek:.1%}")
print(f"A/A false-positive rate looking ONCE     : {false_pos_fixed:.1%}")
assert false_pos_peek > false_pos_fixed, "peeking must inflate false positives"
assert false_pos_peek > 0.10, "20 peeks should blow well past the nominal 5%"
print("\nPeeking inflates false positives well past 5% ✔ — this is why you "
      "pre-commit a sample size (or use a sequential test).")

**The fixes**, in increasing sophistication:

1. **Pre-commit a sample size** and look exactly once. Simple, honest, what we just showed.
2. **Sequential tests** (e.g. mSPRT, group-sequential boundaries, always-valid p-values) that are *designed* to be peeked at continuously without inflating error. Modern experimentation platforms use these.
3. **Separate the two questions.** The p-value answers *"is it better?"* — that can wait for enough samples. But *"is it catastrophically worse right now?"* is a **safety** question, and for that you do NOT wait: you use a fast **guardrail**, which is next.

## 6. The guardrail — burn-rate rollback (reusing Lesson 73)

The significance test is patient. Your users are not. If the challenger is silently melting down, you want it gone in minutes, not after you've collected a statistically clean sample.

That's exactly what Lesson 73's **error budget + burn rate** gives you. Recap:

- An SLO of 95% success over the challenger's traffic implies an **allowed error rate** of 5%.
- **Burn rate** = `observed_error_rate / allowed_error_rate`. A burn rate of `1.0` spends the budget exactly on schedule; `14.4` is Google SRE's classic fast-burn *page-now* threshold.
- For a rollout we make it a **kill switch**: if the *challenger arm's* burn rate crosses the fast threshold on enough samples, **auto-rollback** — no human, no waiting for significance.

We reimplement the two functions compactly here so this notebook is self-contained (in a real repo you'd `from observability.alerting import burn_rate`).

In [ ]:
def error_budget_allowed(slo: float) -> float:
    # allowed error fraction implied by an SLO (e.g. slo=0.95 -> 0.05 allowed)
    return 1.0 - slo

def burn_rate(observed_error_rate: float, slo: float) -> float:
    allowed = error_budget_allowed(slo)
    if allowed <= 0:
        return float("inf")
    return observed_error_rate / allowed

@dataclass
class GuardrailConfig:
    slo: float = 0.95            # promise for the challenger arm
    fast_burn: float = 14.4      # SRE fast-burn page/rollback threshold
    min_samples: int = 150       # don't trip on a handful of unlucky requests

def guardrail_breached(succ_b: int, n_b: int, cfg: GuardrailConfig) -> tuple:
    # returns (breached: bool, observed_burn: float)
    if n_b < cfg.min_samples:
        return False, 0.0        # not enough signal to trust a kill decision
    err = 1.0 - succ_b / n_b
    br = burn_rate(err, cfg.slo)
    return br >= cfg.fast_burn, br

# self-test: the bad challenger (12% error vs 5% allowed -> burn 2.4) does NOT
# trip the 14.4 fast-burn... show that a TRULY broken arm (say 80% error) does.
cfg = GuardrailConfig()
b1, br1 = guardrail_breached(succ_b=880, n_b=1000, cfg=cfg)   # 12% error
b2, br2 = guardrail_breached(succ_b=200, n_b=1000, cfg=cfg)   # 80% error (meltdown)
print(f"12% error -> burn {br1:.1f}, breached={b1}")
print(f"80% error -> burn {br2:.1f}, breached={b2}")
assert not b1 and b2, "fast-burn kill switch fires only on a real meltdown"
print("\nGuardrail OK ✔  (note: a mild regression is caught by the STAT TEST, "
      "a meltdown is caught by the BURN-RATE guardrail — two different jobs.)")

**Two independent safety nets, on purpose.** A *mild* regression (B is a few points worse) won't trip the fast-burn kill switch — and it shouldn't, that would make rollouts flap on noise. The **statistical test** catches the mild-but-real regression when enough samples accumulate. The **burn-rate guardrail** catches the *catastrophic* regression fast, before the stats are even ready. A good controller uses **both**: guardrail first (fast, safety), significance second (patient, decision).

## 7. The staged-ramp rollout controller

Now assemble the pieces. The controller walks a ramp schedule `[0.05, 0.25, 0.50, 1.0]`. At each stage it collects that stage's traffic and makes **one** decision:

- **ROLLBACK** — the guardrail tripped (meltdown), *or* the stat test says B is significantly *worse*. Set split back to 0, stop.
- **PROMOTE** — B is healthy and (for the final push) the evidence supports it; advance to the next stage.
- **HOLD** — not enough signal yet; stay at this stage and gather more (in a real system this is "wait and collect more traffic"; here we just report it).

Because assignment is sticky and the ramp is monotonic, advancing a stage only ever *adds* users to B.

In [ ]:
@dataclass
class StageReport:
    stage: int
    split: float
    n_a: int
    n_b: int
    succ_rate_a: float
    succ_rate_b: float
    burn_b: float
    p_value: float
    decision: str
    reason: str

RAMP = [0.05, 0.25, 0.50, 1.0]

def evaluate_stage(events: list, cfg: GuardrailConfig, stage: int,
                   split: float) -> StageReport:
    a = [e for e in events if e.arm.startswith("A")]
    b = [e for e in events if e.arm.startswith("B")]
    sa, na = sum(e.success for e in a), len(a)
    sb, nb = sum(e.success for e in b), len(b)

    breached, burn = guardrail_breached(sb, nb, cfg)
    # significance test only when both arms have a usable sample
    if na > 30 and nb > 30:
        tr = two_proportion_ztest(sa, na, sb, nb)
        pval, diff, ci_high = tr.p_value, tr.diff, tr.ci_high
    else:
        pval, diff, ci_high = 1.0, 0.0, 1.0

    # ---- decision logic: guardrail (safety) first, then statistics ----
    if breached:
        decision, reason = "ROLLBACK", f"burn-rate {burn:.1f} >= {cfg.fast_burn} (meltdown)"
    elif nb > 30 and pval < 0.05 and diff < 0:
        decision, reason = "ROLLBACK", f"B significantly WORSE (diff {diff:+.3f}, p={pval:.3f})"
    elif nb < cfg.min_samples:
        decision, reason = "HOLD", f"only {nb} B samples (< {cfg.min_samples}) — gather more"
    else:
        decision, reason = "PROMOTE", (
            f"healthy: burn {burn:.1f} ok" +
            (f", B better (diff {diff:+.3f}, p={pval:.3f})" if diff > 0 and pval < 0.05
             else ", no evidence of regression"))
    return StageReport(stage, split, na, nb, sa / na if na else 0.0,
                       sb / nb if nb else 0.0, burn, pval, decision, reason)

def run_rollout(champion: ArmSpec, challenger: ArmSpec, n_per_stage: int = 4000,
                cfg: Optional[GuardrailConfig] = None) -> list:
    cfg = cfg or GuardrailConfig()
    reports = []
    for i, split in enumerate(RAMP):
        events = simulate_stage(champion, challenger, split, n_per_stage, stage_idx=i)
        rep = evaluate_stage(events, cfg, stage=i, split=split)
        reports.append(rep)
        if rep.decision == "ROLLBACK":
            break                      # kill the rollout, champion stays
        # PROMOTE/HOLD -> continue the ramp
    return reports

def show_reports(title: str, reports: list):
    t = Table(title=title, show_lines=False)
    for c in ["Stage", "Split", "nA", "nB", "A succ", "B succ", "B burn", "p-val", "Decision"]:
        t.add_column(c)
    style = {"ROLLBACK": "bold red", "PROMOTE": "bold green", "HOLD": "yellow"}
    for r in reports:
        t.add_row(str(r.stage), f"{r.split:.0%}", str(r.n_a), str(r.n_b),
                  f"{r.succ_rate_a:.3f}", f"{r.succ_rate_b:.3f}", f"{r.burn_b:.1f}",
                  f"{r.p_value:.3f}", f"[{style.get(r.decision,'')}]{r.decision}[/]")
    console.print(t)
    console.print(f"  final reason: [italic]{reports[-1].reason}[/]")

print("Controller defined ✔")

## 8. The payoff — run it on a bad challenger and a good one

Same controller, same thresholds. The only difference is the *true* quality of arm B (which the controller cannot see directly — it only sees noisy samples). The bad challenger should die early; the good one should climb to 100%.

In [ ]:
bad_reports  = run_rollout(CHAMPION, BAD_CHAL)
good_reports = run_rollout(CHAMPION, GOOD_CHAL)

show_reports("Scenario 1 — BAD challenger (must be rolled back)", bad_reports)
print()
show_reports("Scenario 2 — GOOD challenger (should reach 100%)", good_reports)

# ---- assertions: the controller did the right thing in BOTH worlds ----
# Bad challenger: rolled back, and it happened EARLY (not at full traffic).
assert bad_reports[-1].decision == "ROLLBACK", "bad challenger must be rolled back"
assert bad_reports[-1].split <= 0.25, "bad challenger should die at an early, low-traffic stage"
bad_blast = bad_reports[-1].split
print(f"\nBad challenger rolled back at {bad_blast:.0%} traffic — blast radius contained ✔")

# Good challenger: never rolled back, and it completed the ramp to 100%.
assert all(r.decision != "ROLLBACK" for r in good_reports), "good challenger must never roll back"
assert good_reports[-1].split == 1.0 and good_reports[-1].decision == "PROMOTE", \
    "good challenger should be promoted to 100%"
print("Good challenger promoted all the way to 100% ✔")

# 💡 EXPERIMENT: set BAD_CHAL.p_success = 0.945 (only 1.5pt worse). Now the
# burn-rate guardrail won't fire — watch the STAT TEST catch it instead once
# enough samples accumulate at a later stage. Two nets, different jobs.

## 9. Package it — `observability/rollout.py`

Everything reusable goes into a dependency-free module that slots next to `tracing.py` / `online_eval.py` (L71), `logging_search.py` (L72) and `alerting.py` (L73) — the growing `observability/` package headed for the L76 `agent-bench` capstone. Note the module's `decide()` reproduces the guardrail-then-significance logic as one clean entry point.

In [ ]:
# === Write the reusable observability/rollout.py module to disk ===
import os
MODULE_SRC = r"""
# observability/rollout.py
# Guarded-rollout toolkit: sticky bucketing, two-proportion significance test,
# burn-rate guardrail, and a staged-ramp controller. Zero third-party deps.
import math, hashlib
from dataclasses import dataclass
from typing import Optional


def bucket_of(unit_id, salt="exp"):
    # stable hash of unit_id into [0, 1)
    h = hashlib.md5((salt + ":" + str(unit_id)).encode()).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


def assign_arm(unit_id, split, salt="exp"):
    # "B" (challenger) if the unit falls under the split, else "A" (champion)
    return "B" if bucket_of(unit_id, salt) < split else "A"


def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


@dataclass
class TestResult:
    p_a: float
    p_b: float
    diff: float
    z: float
    p_value: float
    ci_low: float
    ci_high: float
    n_a: int
    n_b: int


def two_proportion_ztest(succ_a, n_a, succ_b, n_b, z_crit=1.96):
    p_a = succ_a / n_a
    p_b = succ_b / n_b
    p_pool = (succ_a + succ_b) / (n_a + n_b)
    se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se_pool if se_pool > 0 else 0.0
    p_value = 2 * (1 - normal_cdf(abs(z)))
    se_diff = math.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
    diff = p_b - p_a
    return TestResult(p_a, p_b, diff, z, p_value,
                      diff - z_crit * se_diff, diff + z_crit * se_diff, n_a, n_b)


def error_budget_allowed(slo):
    return 1.0 - slo


def burn_rate(observed_error_rate, slo):
    allowed = error_budget_allowed(slo)
    if allowed <= 0:
        return float("inf")
    return observed_error_rate / allowed


@dataclass
class GuardrailConfig:
    slo: float = 0.95
    fast_burn: float = 14.4
    min_samples: int = 150


def guardrail_breached(succ_b, n_b, cfg):
    if n_b < cfg.min_samples:
        return False, 0.0
    err = 1.0 - succ_b / n_b
    br = burn_rate(err, cfg.slo)
    return br >= cfg.fast_burn, br


@dataclass
class Decision:
    action: str      # PROMOTE | HOLD | ROLLBACK
    reason: str
    split: float
    burn_b: float
    p_value: float


def decide(succ_a, n_a, succ_b, n_b, split, cfg=None):
    # One rollout decision from this stage's counts. Guardrail first (safety),
    # then the significance test (decision).
    cfg = cfg or GuardrailConfig()
    breached, burn = guardrail_breached(succ_b, n_b, cfg)
    if n_a > 30 and n_b > 30:
        tr = two_proportion_ztest(succ_a, n_a, succ_b, n_b)
        pval, diff = tr.p_value, tr.diff
    else:
        pval, diff = 1.0, 0.0
    if breached:
        return Decision("ROLLBACK", "burn-rate meltdown", split, burn, pval)
    if n_b > 30 and pval < 0.05 and diff < 0:
        return Decision("ROLLBACK", "significantly worse", split, burn, pval)
    if n_b < cfg.min_samples:
        return Decision("HOLD", "insufficient samples", split, burn, pval)
    return Decision("PROMOTE", "healthy", split, burn, pval)


DEFAULT_RAMP = [0.05, 0.25, 0.50, 1.0]
"""

path = os.path.join(BASE, 'observability', 'rollout.py')
with open(path, 'w') as f:
    f.write(MODULE_SRC)
print('wrote', path, '(', len(MODULE_SRC), 'bytes )')

# import it back and smoke-test the public surface
import importlib, sys
sys.path.insert(0, BASE)
import observability.rollout as R
importlib.reload(R)
assert R.assign_arm('user_1', 0.5) in ('A', 'B')
_tr = R.two_proportion_ztest(960, 1000, 316, 350)
assert _tr.p_value < 0.05 and _tr.ci_high < 0
assert abs(R.burn_rate(0.05, 0.95) - 1.0) < 1e-9
_d = R.decide(960, 1000, 200, 1000, split=0.05)
assert _d.action == 'ROLLBACK'
print('module smoke-test passed ✔  public API:',
      [n for n in dir(R) if not n.startswith('_')][:8], '...')


## 10. Ten ways guarded rollouts go wrong

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | **Non-sticky assignment** (`random()` per request) | The same user flip-flops between arms; experience is incoherent and the experiment is contaminated. |
| 2 | **Peeking + stop-on-significant** | Inflates false positives far past 5% (§5). Pre-commit a sample size or use a sequential test. |
| 3 | **No guardrail, only the stat test** | A meltdown hurts users for the whole time it takes to reach significance. Fast burn-rate kill switch first. |
| 4 | **Guardrail too twitchy** | `min_samples` too low → rollouts flap and roll back on a handful of unlucky requests. |
| 5 | **Ramping to 100% on the first green stage** | 5% looking fine doesn't mean 100% will. Stage the ramp; each stage is fresh evidence. |
| 6 | **Simpson's paradox / segment mix** | B wins overall but loses in every important segment (mobile, a locale, a request type). Slice your metrics (L72 trace search). |
| 7 | **Metric mismatch** | Optimizing success rate while latency or **cost** silently regresses. Guard *all* the metrics that matter, not just the headline one. |
| 8 | **Sample-ratio mismatch (SRM)** | You asked for 25% B but got 12% — a bucketing/routing bug invalidates the whole test. Always assert the observed split ≈ intended. |
| 9 | **Novelty / primacy effects** | New thing looks great (or bad) for a day because it's *new*, then reverts. Run long enough to outlast the novelty. |
| 10 | **No holdback after 100%** | Ship to everyone and you lose the counterfactual. Keep a small long-term holdback to detect slow regressions. |

## 11. Verification — every claim in this lesson, asserted

A single cell that re-checks the load-bearing facts. If any assertion fails, the lesson is wrong and you should not trust it.

In [ ]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))

# bucketing
check("bucket in [0,1)", 0.0 <= bucket_of("x") < 1.0)
check("assignment sticky", assign_arm("u9", 0.3) == assign_arm("u9", 0.3))
check("ramp monotonic (B@0.1 subset of B@0.4)",
      {u for u in users[:2000] if assign_arm(u, 0.1) == "B"} <=
      {u for u in users[:2000] if assign_arm(u, 0.4) == "B"})
_share = sum(1 for u in users if assign_arm(u, 0.25) == "B") / len(users)
check("split honored (~0.25)", abs(_share - 0.25) < 0.01)

# stat test
_r = two_proportion_ztest(960, 1000, 316, 350)
check("z-test flags real regression (p<0.05)", _r.p_value < 0.05)
check("z-test CI entirely below 0 for worse B", _r.ci_high < 0)
_null = two_proportion_ztest(950, 1000, 951, 1000)
check("z-test does NOT flag identical arms", _null.p_value > 0.05)

# peeking
check("peeking inflates false positives", false_pos_peek > false_pos_fixed)

# guardrail
check("burn_rate(0.05,0.95)==1.0", abs(burn_rate(0.05, 0.95) - 1.0) < 1e-9)
check("guardrail ignores mild regression", not guardrail_breached(880, 1000, GuardrailConfig())[0])
check("guardrail fires on meltdown", guardrail_breached(200, 1000, GuardrailConfig())[0])

# controller end-to-end
check("bad challenger rolled back", bad_reports[-1].decision == "ROLLBACK")
check("bad rollback early (<=25% traffic)", bad_reports[-1].split <= 0.25)
check("good challenger reached 100%", good_reports[-1].split == 1.0)
check("good challenger never rolled back",
      all(r.decision != "ROLLBACK" for r in good_reports))

# module on disk
check("rollout.py written", os.path.exists(os.path.join(BASE, "observability", "rollout.py")))

t = Table(title="Verification checklist", show_lines=False)
t.add_column("#"); t.add_column("Check"); t.add_column("Result")
passed = 0
for i, (name, ok) in enumerate(checks, 1):
    passed += ok
    t.add_row(str(i), name, "[green]PASS[/]" if ok else "[bold red]FAIL[/]")
console.print(t)
print(f"\n{passed}/{len(checks)} checks passed")
assert passed == len(checks), "Some checks failed — see the table above"
print("ALL CHECKS PASSED ✔")

## 12. Summary, homework, and what's next

**What you built today**

| Concept | Primitive | The job it does |
|---|---|---|
| Sticky bucketing | `assign_arm` (hash-based) | Stable, reproducible A/B assignment; monotonic ramps |
| Significance | `two_proportion_ztest` | *Is B really better/worse, or is it noise?* |
| Peeking discipline | A/A simulation | Why you pre-commit a sample size (or go sequential) |
| Guardrail | `burn_rate` + `guardrail_breached` | Fast auto-rollback on a meltdown (reuses L73) |
| Controller | `run_rollout` / `decide` | Staged 5%→100% ramp; promote good, kill bad early |

**The through-line:** two safety nets doing two different jobs — the **guardrail** is fast and about *safety* (catch a meltdown in minutes), the **stat test** is patient and about the *decision* (is it genuinely better). A real rollout uses both.

**Homework (pick one or two)**

1. **Add cost & latency guardrails.** Right now only success rate gates the rollout. Add a p95-latency SLO and a cost-per-request budget; roll back if *either* regresses (pitfall #7).
2. **Sequential test.** Replace fixed-sample significance with an always-valid sequential test (mSPRT or a simple group-sequential alpha-spending boundary) so continuous peeking is legitimate.
3. **Sample-ratio-mismatch alarm.** Add an SRM check that asserts observed split ≈ intended split each stage and rolls back the *experiment* (not the challenger) if a routing bug is detected (pitfall #8).
4. **Segment slicing.** Simulate two user segments where B wins overall but loses on one segment; make the controller refuse to promote until B is non-inferior on *every* segment (Simpson's paradox, pitfall #6).
5. **Wire into `agent-bench`.** Use `agent-bench` (Phase 7) as the offline gate *before* a rollout even starts: only a challenger that beats the champion offline is allowed onto the 5% ramp — connecting offline eval (L49/L61) to online rollout (today).

**Next lesson — L75: Feedback loops & the data flywheel.** You now detect drift (L71), search traces (L72), alert (L73), and roll out safely (L74). L75 closes the loop: how production traffic — the failures you caught, the traces you searched, the arms you compared — becomes *labeled data* that improves your evals and your model. The rollback events and losing arms from today are training signal; we'll build the pipeline that turns them into a compounding advantage.

> Run every cell top-to-bottom in Colab. There's no API key and no network call — the whole lesson is deterministic, so your numbers will match the ones written here exactly.